In [ ]:
!pip install sentence-transformers


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
sentences = [
    # Cricket (4 sentences)
    "The batsman hit a powerful six over mid-wicket to win the match.",
    "India won the Test series after a brilliant bowling performance.",
    "The spinner deceived the batsman with a well-disguised googly.",
    "A dropped catch in the slip cordon proved costly for the fielding side.",
    # Cooking (3 sentences)
    "Saute the onions in olive oil until they turn golden brown.",
    "Marinating the chicken overnight enhances its flavour and tenderness.",
    "Always preheat the oven before placing the cake batter inside.",
    # Cybersecurity (3 sentences)
    "A phishing email tricked the employee into revealing their password.",
    "Firewalls and intrusion detection systems protect networks from attacks.",
    "Encrypting sensitive data ensures it cannot be read if intercepted.",
]

labels = [
    "Cricket-1", "Cricket-2", "Cricket-3", "Cricket-4",
    "Cooking-1", "Cooking-2", "Cooking-3",
    "CyberSec-1", "CyberSec-2", "CyberSec-3",
]

for i, (lbl, s) in enumerate(zip(labels, sentences)):
    print(f"[{i:02d}] {lbl:12s} | {s}")


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(sentences, show_progress_bar=True)
print(f"\nEmbedding shape: {embeddings.shape}")  # Expected: (10, 384)


In [ ]:
cos_sim_matrix = cosine_similarity(embeddings)
print("10x10 Cosine Similarity Matrix:")
print(np.round(cos_sim_matrix, 3))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))

sns.heatmap(
    cos_sim_matrix,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    xticklabels=labels,
    yticklabels=labels,
    linewidths=0.5,
    linecolor="white",
    vmin=0, vmax=1,
    ax=ax
)

# Topic boundary lines
for boundary in [4, 7]:
    ax.axhline(boundary, color="black", linewidth=2.5)
    ax.axvline(boundary, color="black", linewidth=2.5)

ax.set_title("10×10 Cosine Similarity Matrix\n(Cricket | Cooking | Cybersecurity)",
             fontsize=14, fontweight="bold", pad=15)
plt.xticks(rotation=35, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig("cosine_similarity_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
query = "The bowler took three wickets in one over"
query_embedding = model.encode([query])
query_sims = cosine_similarity(query_embedding, embeddings)[0]
top2 = np.argsort(query_sims)[::-1][:2]

print(f'Query: "{query}"\n')
print(f"{'Rank':<6} {'Label':<14} {'Score':<8} Sentence")
print("-" * 70)
for rank, idx in enumerate(top2, 1):
    print(f"{rank:<6} {labels[idx]:<14} {query_sims[idx]:.4f}   {sentences[idx]}")

print("\nAll scores:")
print("-" * 70)
for i, (lbl, score) in enumerate(zip(labels, query_sims)):
    bar = "█" * int(score * 30)
    print(f"[{i:02d}] {lbl:12s} | {score:.4f} | {bar}")